# Data Pipeline

## preparation

installs

In [16]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [38]:
%pip install pinecone langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


imports

In [45]:
import os
from dotenv import load_dotenv, find_dotenv
from pathlib import Path
import re
import json
from openai import OpenAI
import pandas as pd
import numpy as np

from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
env_path = Path().resolve().parents[3] / ".env"

load_dotenv(find_dotenv())

api_key = os.getenv("OPENAI_API_KEY")

# Notebook summary

### PREPROCESSING PIPELINE

- Download transcripts
- Dataset cleaning
    - Delete timestamps
    - Context marks
    - Noises
    - Non dialogue text
    - Identify speakers (detective, suspect)
- Create detective VDB
    - Filter detective questions
    - Create synthetic question
    - Deduplicate based on cosine similarity
- Create suspect VDB



### CREATE DATABASES

- Vector DB 1 detective_questions
Purpose: evaluate player questions (contains only detective)
{
"id": "lazarus_q_034",
"text": "Did you ever meet his wife?",
"case_id": "lazarus",
"phase": "narrative",
"question_type": "relationship"
}

- Vector DB 2 suspect_chunks
Purpose: generate realistic answers by suspect (contains question + answer)
{
"id": "lazarus_chunk_014",
"case_id": "lazarus",
"phase": "narrative",
"embedding_text": "...Detective: ... Suspect: ..."
}

- DB 3
case_facts
Purpose: aux DB to keep consistency in different part of the system

### General preprocessing

raw transcript
→ remove timestamps
→ speaker segmentation


In [ ]:
# source of transcript: youtube.com/t5ljpPTNvCM

In [ ]:
# load transcript raw

file_path = "../data/transcripts_raw/t5ljpPTNvCM.txt"
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Raw transcript length:", len(raw_text))

Raw transcript length: 63111


delete timestamps and spaces

In [ ]:
# delete timestamps

timestamp_pattern = r"\(\d{1,2}:\d{2}(?::\d{2})?\)"
clean_text = re.sub(timestamp_pattern, "", raw_text)

# delete double spaces
clean_text = re.sub(r"\s+", " ", clean_text).strip()
print("Text length after timestamp removal:", len(clean_text))


Text length after timestamp removal: 61725


In [ ]:
# export cleaned transcript
output_path = "../data/transcripts_clean/t5ljpPTNvCM_clean.txt"
Path("../data/transcripts_clean").mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(clean_text)

print("Clean transcript saved to:", output_path)

Clean transcript saved to: ../data/transcripts_clean/t5ljpPTNvCM_clean.txt


assign speakers

In [12]:
# load clean transcript 
input_path = "../data/transcripts_clean/t5ljpPTNvCM_clean.txt"

with open(input_path, "r", encoding="utf-8") as f:
    clean_text = f.read()

print("Loaded clean transcript length:", len(clean_text))

Loaded clean transcript length: 61725


In [13]:
client = OpenAI()

# split text
def split_text(text, chunk_size=5000):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks


chunks = split_text(clean_text)

print("Total chunks:", len(chunks))


# process chunks
segmented_chunks = []

for i, chunk in enumerate(chunks):

    print(f"Processing chunk {i+1}/{len(chunks)}")

    prompt = f"""
You are analyzing a police interrogation transcript.

Segment the dialogue and label each segment as either:

Detective:
Suspect:

Keep the original text exactly.

Example format:

Detective: ...
Suspect: ...
Detective: ...
Suspect: ...

Transcript:
{chunk}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    segmented_text = response.choices[0].message.content

    segmented_chunks.append(segmented_text)


# combine results
segmented_transcript = "\n".join(segmented_chunks)

print("\nPreview:\n")
print(segmented_transcript[:1000])




Total chunks: 13
Processing chunk 1/13
Processing chunk 2/13
Processing chunk 3/13
Processing chunk 4/13
Processing chunk 5/13
Processing chunk 6/13
Processing chunk 7/13
Processing chunk 8/13
Processing chunk 9/13
Processing chunk 10/13
Processing chunk 11/13
Processing chunk 12/13
Processing chunk 13/13

Preview:

Detective: Uh, I'm not sure. Stephanie, I don't know if you know my partner. Great. Hi, St. Nice to meet you. Nice to see you guys. How's it going? Milling around. Uh, well, have a seat. What this is is a I didn't want to bring this up in your squadron. 

Suspect: Oh, no. It's fine. I think we're good to We're bring You're going to bring somebody in, right? 

Detective: Yeah. Well, explain the whole thing, too. I don't want to talk about this in the squadron because I don't know who people are listening. And if we go to my side, everybody's always wondering what everybody else is doing. 

Suspect: Okay. 

Detective: But uh like we're talking about being busy and stuff, we'v

In [14]:
#export

output_path = "../data/transcripts_segmented/t5ljpPTNvCM_segmented.txt"

Path("../data/transcripts_segmented").mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(segmented_transcript)

print("\nSegmented transcript saved to:", output_path)


Segmented transcript saved to: ../data/transcripts_segmented/t5ljpPTNvCM_segmented.txt


In [ ]:
# convert to csv

input_path = "../data/transcripts_segmented/t5ljpPTNvCM_segmented.txt"

with open(input_path, "r", encoding="utf-8") as f:
    lines = f.readlines()
segments = []

for line in lines:
    line = line.strip()
    if not line:
        continue
    if line.startswith("Detective:"):
        speaker = "detective"
        text = line.replace("Detective:", "").strip()
    elif line.startswith("Suspect:"):
        speaker = "suspect"
        text = line.replace("Suspect:", "").strip()
    else:
   
        speaker = "unknown"
        text = line
    segments.append({
        "speaker": speaker,
        "text": text
    })

print("Segments parsed:", len(segments))

# df
df = pd.DataFrame(segments)

# segment id
df.insert(0, "segment_id", [f"seg_{i:04}" for i in range(len(df))])

Segments parsed: 489


In [ ]:

# export csv

output_path = "../data/transcripts_segmented/t5ljpPTNvCM_segments_review.csv"
Path("../data/transcripts_segmented").mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print("CSV saved to:", output_path)
df.head(10)

CSV saved to: ../data/transcripts_segmented/t5ljpPTNvCM_segments_review.csv


,segment_id,speaker,text
0,seg_0000,detective,"Uh, I'm not sure. Stephanie, I don't know if y..."
1,seg_0001,suspect,"Oh, no. It's fine. I think we're good to We're..."
2,seg_0002,detective,"Yeah. Well, explain the whole thing, too. I do..."
3,seg_0003,suspect,Okay.
4,seg_0004,detective,But uh like we're talking about being busy and...
5,seg_0005,suspect,Okay.
6,seg_0006,detective,It's a new case and reviewing the case. There'...
7,seg_0007,suspect,Okay. Do you know John Ruten?
8,seg_0008,detective,John. John Rutton. Rutton. Rutton.
9,seg_0009,suspect,"Oh yeah, I went to school with him."


In [ ]:
# load reviewed csv
input_path = "../data/transcripts_segmented/t5ljpPTNvCM_segments_reviewed.csv"
df = pd.read_csv(input_path, sep=";", engine="python")
print("Rows loaded:", len(df))



Rows loaded: 465


Base file for VDBs

In [ ]:

# convert to json

# remove nan
df = df.where(pd.notnull(df), None)

# convert to dict
segments = df.to_dict(orient="records")
print("Segments ready:", len(segments))

# export JSON
output_path = "../data/transcripts_segmented/t5ljpPTNvCM_segments_clean.json"
Path("../data/transcripts_segmented").mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(segments, f, indent=2, ensure_ascii=False)
print("JSON saved to:", output_path)

# preview
for seg in segments[:10]:
    print(seg)

Segments ready: 465
JSON saved to: ../data/transcripts_segmented/t5ljpPTNvCM_segments_clean.json
{'segment_id': 'seg_0000', 'speaker': 'detective', 'text': 'Nice to meet you. Uh, well, have a seat. ', 'phase': 0}
{'segment_id': 'seg_0001', 'speaker': 'suspect', 'text': "Nice to see you guys. How's it going? What is this? Is this a…", 'phase': 0}
{'segment_id': 'seg_0002', 'speaker': 'detective', 'text': "I didn't want to bring this up in your squadron.", 'phase': 0}
{'segment_id': 'seg_0003', 'speaker': 'suspect', 'text': "Oh, no. It's fine. I think we're good.", 'phase': 0}
{'segment_id': 'seg_0004', 'speaker': 'detective', 'text': "Yeah. Well, explain the whole thing, too. I don't want to talk about this in the squadron because I don't know who people are listening. And if we go to my side, everybody's always wondering what everybody else is doing.", 'phase': 0}
{'segment_id': 'seg_0005', 'speaker': 'suspect', 'text': 'Okay.', 'phase': 0}
{'segment_id': 'seg_0006', 'speaker': 'detect

## VDB detective_questions

VDB-specific preprocessing

- detective segments + metadata = detective_raw_lazarus.json
- manual review
   - delete mistakes
   - assign phase
   - assign pressure
- generate synthetic variants
- final detective VDB

In [18]:
# load segments
input_path = "../data/transcripts_segmented/t5ljpPTNvCM_segments_clean.json"

with open(input_path, "r", encoding="utf-8") as f:
    segments = json.load(f)

print("Segments loaded:", len(segments))

Segments loaded: 465


In [19]:
# filter for detective
detective_segments = [
    s for s in segments
    if s["speaker"] == "detective"
]

print("Detective segments:", len(detective_segments))


Detective segments: 193


In [ ]:
"""
# clean fillers

fillers = [
    r"\bwell\b",
    r"\bi mean\b",
    r"\byou know\b",
    r"\blet me ask you\b",
    r"\bkind of\b",
    r"\bbasically\b"
]
"""

BUILD DATESET

In [20]:
case_id = "lazarus"

dataset = []

for seg in detective_segments:

    dataset.append({
        "id": f"{case_id}_{seg['segment_id']}",
        "segment_id": seg["segment_id"],
        "text": seg["text"].strip(),
        "phase": int(seg["phase"]),
        "source": "original"
    })


print("Dataset size:", len(dataset))

Dataset size: 193


In [ ]:
# export json

output_path = "../data/datasets/detective_lazarus_original.json"
Path("../data/datasets").mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
print("Saved to:", output_path)

# preview
for x in dataset[:10]:
    print(x)

Saved to: ../data/datasets/detective_lazarus_original.json
{'id': 'lazarus_seg_0000', 'segment_id': 'seg_0000', 'text': 'Nice to meet you. Uh, well, have a seat.', 'phase': 0, 'source': 'original'}
{'id': 'lazarus_seg_0002', 'segment_id': 'seg_0002', 'text': "I didn't want to bring this up in your squadron.", 'phase': 0, 'source': 'original'}
{'id': 'lazarus_seg_0004', 'segment_id': 'seg_0004', 'text': "Yeah. Well, explain the whole thing, too. I don't want to talk about this in the squadron because I don't know who people are listening. And if we go to my side, everybody's always wondering what everybody else is doing.", 'phase': 0, 'source': 'original'}
{'id': 'lazarus_seg_0006', 'segment_id': 'seg_0006', 'text': "But uh like we're talking about being busy and stuff, we've been assigned a case that we've been looking at.", 'phase': 0, 'source': 'original'}
{'id': 'lazarus_seg_0008', 'segment_id': 'seg_0008', 'text': "It's a new case and reviewing the case. There's some notes to see a

In [ ]:
# export in csv

# load from json
input_path = "../data/datasets/detective_lazarus_original.json"
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)
df = pd.DataFrame(data)
print("Rows loaded:", len(df))

# export csv 
csv_path = "../data/datasets/detective_raw_lazarus.csv"
df.to_csv(csv_path, index=False)
print("CSV exported to:", csv_path)
df.head()

Rows loaded: 193
CSV exported to: ../data/datasets/detective_raw_lazarus.csv


,id,segment_id,text,phase,source
0,lazarus_seg_0000,seg_0000,"Nice to meet you. Uh, well, have a seat.",0,original
1,lazarus_seg_0002,seg_0002,I didn't want to bring this up in your squadron.,0,original
2,lazarus_seg_0004,seg_0004,"Yeah. Well, explain the whole thing, too. I do...",0,original
3,lazarus_seg_0006,seg_0006,But uh like we're talking about being busy and...,0,original
4,lazarus_seg_0008,seg_0008,It's a new case and reviewing the case. There'...,0,original


CREATE SYNTHETHIC QUESTIONS

In [ ]:
#load json

input_path = "../data/datasets/detective_lazarus_original.json"
with open(input_path, "r", encoding="utf-8") as f:
    originals = json.load(f)

print("Original questions:", len(originals))

Original questions: 193


In [30]:
# generate synthethic questions

client = OpenAI()
synthetic_examples = []
counter = 0
N_VARIANTS = 10
for item in originals:

    text = item["text"]
    phase = item["phase"]
    segment_id = item["segment_id"]

    prompt = f"""
You are helping generate training data for a police interrogation simulation.

Each question belongs to a specific interrogation phase.
Your rewrites MUST keep the same level of pressure and strategy.

Interrogation phases:

0 — Small talk / rapport building.
Friendly tone. No suspicion.

1 — Narrative.
Ask about memories, timeline, relationships with John.

2 — General victim discussion.
Talk about the victim in a neutral way.

3 — Evidence trap.
Ask specific questions about the victim without revealing evidence.

4 — Confrontation.
Mention testimonies from others, inconsistencies, or facts that contradict the suspect's statements.

5 — Collapse.
Direct pressure. Mention DNA or other strong evidence, legalconsequences, reading Miranda rights.

Original question:
{text}

Phase: {phase}

Rewrite the question in {N_VARIANTS} different ways.

Use different linguistic strategies so the sentences are structurally different:

- direct question
- timeline framing
- indirect phrasing
- clarification question
- conversational tone
- alternative wording

Rules:
- Keep the same meaning.
- Keep the same interrogation strategy.
- Keep the same pressure level of the phase.
- Do not mention the phase in the output.

Return exactly {N_VARIANTS} lines.
Each line must contain one rewritten question.
Do not number them.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.8,
        messages=[{"role": "user", "content": prompt}]
    )

    variants = response.choices[0].message.content.strip().split("\n")

    # ensure number of variants
    variants = variants[:N_VARIANTS]

    for v in variants:

        v = v.strip()

        if not v:
            continue

        counter += 1

        synthetic_examples.append({
            "id": f"syn_{counter:06}",
            "segment_id": segment_id,
            "text": v,
            "phase": phase,
            "source": "synthetic"
        })

print("Synthetic examples generated:", len(synthetic_examples))

Synthetic examples generated: 1785


Combine original + synthetic dataset

In [33]:
# combine dataset
full_dataset = originals + synthetic_examples
print("Original:", len(originals))
print("Synthetic:", len(synthetic_examples))
print("Total:", len(full_dataset))

Original: 193
Synthetic: 1785
Total: 1978


DATASET FOR VDB

In [34]:
# export final dataset
output_path = "../data/datasets/detective_dataset_lazarus_full.json"

Path("../data/datasets").mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(full_dataset, f, indent=2, ensure_ascii=False)
print("Dataset saved to:", output_path)

Dataset saved to: ../data/datasets/detective_dataset_lazarus_full.json


PINECONE VDB

Deduplicate embeddings that are too similar



In [46]:
# create embeddings

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
texts = [item["text"] for item in full_dataset]
print("Creating embeddings...")
vectors = embeddings_model.embed_documents(texts)
vectors = np.array(vectors)
print("Embedding matrix shape:", vectors.shape)



Creating embeddings...
Embedding matrix shape: (1978, 1536)


In [56]:
# detect duplicates

threshold = 0.90   # similarity threshold

vectors_np = np.array(vectors)
similarity_matrix = cosine_similarity(vectors_np)
keep_indices = []
removed_indices = set()

for i in range(len(vectors_np)):

    if i in removed_indices:
        continue

    keep_indices.append(i)

    for j in range(i + 1, len(vectors_np)):

        if similarity_matrix[i, j] > threshold:
            removed_indices.add(j)

print("Threshold:", threshold)
print("Original size:", len(full_dataset))
print("Would remove:", len(removed_indices))
print("Would remain:", len(keep_indices))

Threshold: 0.9
Original size: 1978
Would remove: 54
Would remain: 1924


In [57]:
# inspect duplicates
pairs = []

for i in range(len(vectors_np)):
    for j in range(i + 1, len(vectors_np)):
        if similarity_matrix[i, j] > threshold:
            pairs.append((i, j, similarity_matrix[i, j]))

print("Duplicate pairs found:", len(pairs))

for i, j, sim in pairs[:10]:
    print("\nSIM:", round(sim, 3))
    print("A:", full_dataset[i]["text"])
    print("B:", full_dataset[j]["text"])

Duplicate pairs found: 76

SIM: 1.0
A: Yeah.
B: Yeah.

SIM: 1.0
A: Oh, really?
B: Oh, really?

SIM: 0.918
A: How would you describe your relationship with John?
B: So, how would you describe your connection with John?

SIM: 0.913
A: What was your friendship with John like?
B: What was your friendship with John like? Did it ever develop into anything beyond that?

SIM: 0.923
A: I'm curious about your relationship with John and how it developed over time.
B: Can you tell me about how your relationship with John developed over time?

SIM: 0.97
A: Can you explain your relationship with John a bit more?
B: Can you explain the nature of your relationship with John a bit more?

SIM: 0.907
A: Can you explain your relationship with John a bit more?
B: Could you tell me a bit about your relationship with John?

SIM: 0.908
A: Can you tell me what your living situation was like with John in the dorm?
B: How would you describe your living arrangements with John in the dorm?

SIM: 0.931
A: Can you t

In [58]:
# execute remove duplicates
clean_dataset = [full_dataset[i] for i in keep_indices]
print("Clean dataset size:", len(clean_dataset))

output_path = "../data/datasets/detective_dataset_lazarus_dedup.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(clean_dataset, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

Clean dataset size: 1924
Saved to: ../data/datasets/detective_dataset_lazarus_dedup.json


In [61]:
# load dateset

with open("../data/datasets/detective_dataset_lazarus_dedup.json", encoding="utf-8") as f:
    dataset = json.load(f)

print("Dataset size:", len(dataset))


Dataset size: 1924


In [78]:
# CREATE INDEX

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "detective-questions"

if index_name not in [i.name for i in pc.list_indexes()]:
    
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index_detective = pc.Index(index_name)

In [41]:
# CREATE EMBEDDINGS

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [69]:
# INSERT DATA IN PINECONE 

#prepare text and metadata
texts = [item["text"] for item in dataset]

metadatas = [
    {
        "phase": item["phase"],
        "source": item["source"],
        "segment_id": item["segment_id"]
    }
    for item in dataset
]

ids = [item["id"] for item in dataset]

In [80]:
from langchain_pinecone import PineconeVectorStore

#insert
vector_store_detective = PineconeVectorStore(
    index=index_detective,
    embedding=embeddings
)

vector_store.add_texts(
    texts=texts,
    metadatas=metadatas,
    ids=ids
)

print("Inserted vectors:", len(texts))

Inserted vectors: 167


 Test retrieval


In [71]:
query = "When was the last time you saw John?"

results = vector_store.similarity_search(query, k=5)

for r in results:
    print(r.page_content, r.metadata)

When was the last time you saw John, and what was happening then? {'phase': 1.0, 'segment_id': 'seg_0046', 'source': 'synthetic'}
Can you walk me through the events leading up to when you last saw John? {'phase': 1.0, 'segment_id': 'seg_0025', 'source': 'synthetic'}
Can you tell me what you remember about your last interactions with John? {'phase': 1.0, 'segment_id': 'seg_0046', 'source': 'synthetic'}
Do you recall any other people who interacted with John recently? {'phase': 3.0, 'segment_id': 'seg_0256', 'source': 'synthetic'}
Thinking back, can you tell me about the times you’ve spent with John? {'phase': 1.0, 'segment_id': 'seg_0244', 'source': 'synthetic'}


## VDB suspect_behaviour

In [ ]:
# Load cleaned transcript segments
input_path = "../data/transcripts_segmented/t5ljpPTNvCM_segments_clean.json"
with open(input_path, "r", encoding="utf-8") as f:
    segments = json.load(f)
print("Segments loaded:", len(segments))

Chunk creation

In [ ]:
# create interrogation turns
# turn = Detective question + suspect response

turns = []

i = 0
while i < len(segments):

    seg = segments[i]
    if seg["speaker"] == "detective":

        detective_text = seg["text"].strip()
        phase = int(seg["phase"])
        i += 1
        suspect_responses = []

        # collect all suspect responses until next detective
        while i < len(segments) and segments[i]["speaker"] == "suspect":
            suspect_responses.append(segments[i]["text"].strip())
            i += 1

        # merge suspect responses
        suspect_text = " ".join(suspect_responses)
        if suspect_text:
            turns.append({
                "detective": detective_text,
                "suspect": suspect_text,
                "phase": phase
            })

    else:
        i += 1

print("Turns reconstructed:", len(turns))



# Create sliding windows of 3 turns
# This preserves conversational context

WINDOW_SIZE = 3

chunks = []

for i in range(len(turns) - WINDOW_SIZE + 1):

    window = turns[i:i + WINDOW_SIZE]

    lines = []

    for t in window:
        lines.append(f"Detective: {t['detective']}")
        lines.append(f"Suspect: {t['suspect']}")

    embedding_text = "\n".join(lines)

    chunks.append({
        "id": f"chunk_{i:04}",
        "phase": window[-1]["phase"],  # phase of the last turn
        "embedding_text": embedding_text,
        "turns": window
    })

print("Chunks created:", len(chunks))



# Export dataset

output_path = "../data/datasets/suspect_chunks_lazarus.json"
Path("../data/datasets").mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)
print("Dataset saved to:", output_path)


# Preview chunks
for c in chunks[:3]:
    print("\n--- CHUNK ---")
    print(c["embedding_text"])

Turns reconstructed: 169
Chunks created: 167
Dataset saved to: ../data/datasets/suspect_chunks_lazarus.json

--- CHUNK ---
Detective: Nice to meet you. Uh, well, have a seat.
Suspect: Nice to see you guys. How's it going? What is this? Is this a…
Detective: I didn't want to bring this up in your squadron.
Suspect: Oh, no. It's fine. I think we're good.
Detective: Yeah. Well, explain the whole thing, too. I don't want to talk about this in the squadron because I don't know who people are listening. And if we go to my side, everybody's always wondering what everybody else is doing.
Suspect: Okay.

--- CHUNK ---
Detective: I didn't want to bring this up in your squadron.
Suspect: Oh, no. It's fine. I think we're good.
Detective: Yeah. Well, explain the whole thing, too. I don't want to talk about this in the squadron because I don't know who people are listening. And if we go to my side, everybody's always wondering what everybody else is doing.
Suspect: Okay.
Detective: But uh like we're

In [73]:
# load chunks

with open("../data/datasets/suspect_chunks_lazarus.json", encoding="utf-8") as f:
    suspect_chunks = json.load(f)

print("Chunks loaded:", len(suspect_chunks))

Chunks loaded: 167


In [74]:
# prepare text and metadata

texts = [c["embedding_text"] for c in suspect_chunks]

metadatas = [
    {
        "phase": c["phase"],
        "turn_count": len(c["turns"]),
        "type": "suspect_behaviour"
    }
    for c in suspect_chunks
]

ids = [c["id"] for c in suspect_chunks]

print("Texts prepared:", len(texts))

Texts prepared: 167


In [81]:
# create index

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "suspect-behaviour"

existing = [i.name for i in pc.list_indexes()]

if index_name not in existing:

    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index_suspect = pc.Index(index_name)

print("Index ready")

Index ready


In [ ]:
# create embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [82]:
# connect pinecone with langchain
vector_store_suspect = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [83]:
# insert chunks in index 
vector_store_suspect.add_texts(
    texts=texts,
    metadatas=metadatas,
    ids=ids
)

print("Inserted chunks:", len(texts))

Inserted chunks: 167


In [84]:
# test retrieval
query = "Did you ever meet Sherri?"

results = vector_store_suspect.similarity_search(query, k=5)

for r in results:
    print("\n---")
    print(r.page_content)
    print(r.metadata)


---
Detective: Um, I mean, when when you heard about uh John's wife being killed, I mean, what was your what was your reaction? I mean, did you thought you heard about it what through a friend or in a bulletin or something?
Suspect: Um I obviously I mean I called I called the family. Um I I called maybe some of his friends that that that I knew and I mean obviously it shocked you if you're if I heard it at work, you know. Um which I may have I I faintly remember a bulletin going around. Um either that or somebody called me. I I also don't remember. Um uh and then I called probably called his family. um called uh I don't initially I can't say if I initially spoke to him or not. I honestly don't remember. I may have said to somebody, "Hey, have him call me if he wants to talk." And then he may have done that.
Detective: Um you know, um do you know what the circumstances regarding her death?
Suspect: Jeez, let me think back. Um, jeez. I don't know if it was, you know, if it was a burglar